In [5]:
import re
import pandas as pd
import numpy as np

def parse_poker_log_improved(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    games = re.split(r'Game started at:.*?\n', content)[1:]
    results = []
    for game in games:
        game_id_match = re.search(r'Game ID: (\d+)', game)
        game_id = game_id_match.group(1) if game_id_match else None
        pot_match = re.search(r'Pot: ([\d\.]+)', game)
        if pot_match:
            pot_str = pot_match.group(1).rstrip('.')
            try:
                pot_size = float(pot_str)
            except ValueError:
                pot_size = 0
        else:
            pot_size = 0
        players = re.findall(r'Seat \d+: (\w+) \([\d\.]+\)', game)
        num_players = len(players)
        has_flop = 1 if '*** FLOP ***' in game else 0
        has_turn = 1 if '*** TURN ***' in game else 0
        has_river = 1 if '*** RIVER ***' in game else 0
        has_raise = 1 if 'raises' in game else 0
        has_all_in = 1 if 'all-in' in game.lower() else 0
        did_show = 1 if 'shows:' in game else 0

        winner_match = re.search(r'\*Player (\w+).*?Collects: ([\d\.]+)', game)
        winner = winner_match.group(1) if winner_match else None
        is_bluff = 0
        if did_show and winner:
            show_match = re.search(rf'\*?Player {winner} shows:.*?\[(.*?)\]', game)
            if show_match:
                cards = show_match.group(1)
    
                has_high_cards = any(card in cards for card in ['J', 'Q', 'K', 'A'])

                if not has_high_cards and 'pair' not in game.lower():
                    is_bluff = 1
            else:

                is_bluff = 0

        elif winner and 'mucks' in game and 'does not show' in game:
            bluff_score = 0
            if has_raise:
                bluff_score += 1
            if has_flop:
                bluff_score += 1
            if not has_all_in:
                bluff_score += 1
            if bluff_score >= 2:
                is_bluff = 1
        else:
            is_bluff = 0
        
        results.append({
            'game_id': game_id,
            'pot_size': pot_size,
            'num_players': num_players,
            'has_flop': has_flop,
            'has_turn': has_turn,
            'has_river': has_river,
            'has_raise': has_raise,
            'has_all_in': has_all_in,
            'did_show': did_show,
            'is_bluff': is_bluff,
            'winner': winner
        })
    
    return pd.DataFrame(results)
df = parse_poker_log_improved('Export Holdem Manager.txt')
df.to_csv('poker_data_improved.csv', index=False)

print(f"Обработано {len(df)} раздач")
print(f" Средний банк: ${df['pot_size'].mean():.2f}")
print(f"\nРаспределение блефов:")
print(f"   Не блеф (0): {(df['is_bluff']==0).sum()} раздач ({df['is_bluff'].value_counts(normalize=True)[0]*100:.1f}%)")
print(f"   Блеф (1):   {(df['is_bluff']==1).sum()} раздач ({df['is_bluff'].value_counts(normalize=True)[1]*100:.1f}%)")

print("\nПервые 5 строк:")
df.head()

Обработано 2468 раздач
 Средний банк: $11.50

Распределение блефов:
   Не блеф (0): 689 раздач (27.9%)
   Блеф (1):   1779 раздач (72.1%)

Первые 5 строк:


,game_id,pot_size,num_players,has_flop,has_turn,has_river,has_raise,has_all_in,did_show,is_bluff,winner
0,787027613,2.50,9,0,0,0,1,0,0,1,dankmann
1,787027929,24.95,9,1,1,1,1,0,1,0,StephCurry
2,787027464,2.50,9,0,0,0,1,0,0,1,AironVega
3,787027410,6.18,6,1,1,0,1,0,0,1,aleks0v
4,787027157,5.00,6,0,0,0,1,0,0,1,Sephiroth1
